In [ ]:
\/ GENERATOR \/

In [33]:
"""
Генератор синтетики v4.

Ключевые изменения относительно v3:
- Больше НЕ создаём пары (X, rot(X)) из одного изображения.
  Каждый пример генерируется независимо со случайной меткой.
- Жёсткий поворот снижен до ±5-8° (было 15-30°).
- Добавлены тёмные фоны (30% примеров), а не только светлые.
- Рендер на большом фоне + обрезка с рандомными асимметричными полями
  (имитация bbox детектора, а не плотный кроп текста).
- Обрезка краёв независимая с каждой стороны (0-10%).
- Смягчены пороги is_readable — не отбрасываем низкоконтрастные кропы.
- get_bg_color берёт цвет из угла, а не медиану по краям.
- Поворот 180° делается ПОСЛЕ аугментаций, но примеры независимы.
"""

import io
import random
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance
from tqdm import tqdm
from fontTools.ttLib import TTFont


SEED = 2026
random.seed(SEED)
np.random.seed(SEED)


# ============================================================
# НАСТРОЙКИ
# ============================================================
N_SAMPLES = 70000
OUT_DIR = "data/v4"

ENABLE_INVERSION    = True    # теперь это выбор тёмной палитры, а не 255-img
ENABLE_PHOTOMETRIC  = True
ENABLE_ROTATION     = True    # ±3° мягкий наклон
ENABLE_COLOR_BG     = True
ENABLE_COLOR_TEXT   = True
ENABLE_NOISE        = True
ENABLE_GRADIENT     = True
ENABLE_SOFT_BLUR    = True
ENABLE_HARD_ROTATION = True   # теперь ±5-8°, а не 15-30°
ENABLE_CROP         = True    # обрезка 0-10% независимо с каждой стороны

P_HARD_ROTATION = 0.25        # доля примеров с наклоном ±5-8°
P_DARK_BG       = 0.30        # доля примеров с тёмным фоном
# ============================================================


# ============================================================
# КОРПУС
# ============================================================
RU_WORDS = [
    "магазин", "аптека", "метро", "магнит", "лента",
    "москва", "россия", "город", "улица", "дом",
    "доставка", "бесплатно", "скидка", "акция",
    "телефон", "интернет", "кафе", "ресторан",
    "открыто", "закрыто", "выход", "вход", "парковка",
    "детская", "одежда", "обувь", "сумки",
    "молоко", "хлеб", "сыр", "мясо", "рыба",
    "здоровье", "красота", "спорт", "фитнес",
    "школа", "университет", "библиотека", "музей",
    "реклама", "объявление", "новости", "погода",
    "поезд", "самолёт", "такси", "автобус",
    "деньги", "банк", "карта", "кредит",
]
EN_WORDS = [
    "store", "shop", "market", "metro", "mall", "cafe",
    "open", "closed", "exit", "entrance", "parking",
    "sale", "discount", "free", "delivery", "order",
    "phone", "internet", "bank", "credit",
    "food", "drink", "pizza", "coffee",
    "hotel", "restaurant", "bar", "gym",
    "doctor", "clinic", "pharmacy", "hospital",
    "school", "university", "library", "museum",
    "news", "weather", "sport", "game", "music",
    "train", "plane", "taxi", "bus",
    "style", "model", "code", "premium",
]
BRANDS = [
    "DIOR", "PRADA", "GUCCI", "NIKE", "ADIDAS", "PUMA",
    "SAMSUNG", "APPLE", "XIAOMI", "HUAWEI", "SONY",
    "SANYO", "PHILIPS", "BOSCH", "MAKITA",
    "MAZDA", "TOYOTA", "HONDA", "NISSAN", "BMW",
    "PRINCE", "MAYA", "TENZEL", "HIGHWAY", "OVERLIMIT",
]
MIXED = [
    "ГАЗ", "МТС", "БИЛАЙН", "МЕГАФОН", "ТНТ",
    "ХАН", "МИР", "ДОМ", "СВЕТ", "ОАО",
]
PHRASES = [
    "Бесплатная доставка", "Только сегодня",
    "Большой выбор", "Открыто с 9 до 21",
    "По вопросам зачисления", "Включите автовыплату",
    "Поздравляем с покупкой", "Скидка на весь ассортимент",
]
HARD_STRINGS = [
    # Цифры с асимметрией (хвостики, разные изгибы сверху/снизу)
    "505",  "202", "303", "404",  "707",
    "123", "321", "456", "654", "789", "987",

    # Палиндромы с асимметричными буквами (засечки, разная толщина)
    "МАМА", "ДОМ", "МОДА", "ПОП", "ТОТ", "ДОВОД",
    "БОБ", "ДЕД", "ШАЛАШ", "КАЗАК",

    # Латиница с чёткой вертикальной асимметрией
    "top", "lot", "pot", "dot", "mom", "dad", "pop", "tot",
    "level", "radar",

    # Смешанные: цифра + буква, цифра даёт явный признак ориентации
    "А1", "1А", "Б2", "2Б", "В3", "3В", "Г4", "4Г",
]


def sample_text(rng):
    kind = rng.choice(
        ["ru", "en", "brand", "mixed", "phrase", "hard", "digits"],
        p=[0.22, 0.20, 0.15, 0.08, 0.10, 0.15, 0.10],
    )
    if kind == "ru":
        return rng.choice(RU_WORDS)
    if kind == "en":
        return rng.choice(EN_WORDS)
    if kind == "brand":
        return rng.choice(BRANDS)
    if kind == "mixed":
        return rng.choice(MIXED)
    if kind == "phrase":
        return rng.choice(PHRASES)
    if kind == "hard":
        return rng.choice(HARD_STRINGS)
    # digits
    n = int(rng.integers(2, 10))
    return "".join(str(int(rng.integers(0, 10))) for _ in range(n))


# ============================================================
# ЦВЕТА
# ============================================================
LIGHT_BG_PALETTE = [
    (255, 255, 255), (248, 248, 250), (250, 245, 240),
    (200, 230, 255), (255, 200, 220), (200, 255, 210),
    (255, 230, 180), (220, 200, 255), (255, 220, 200),
    (180, 230, 240), (230, 255, 200), (255, 240, 200),
    (210, 230, 255), (245, 210, 240),
]

DARK_BG_PALETTE = [
    (0, 0, 0), (20, 20, 30), (40, 20, 20), (20, 40, 20),
    (20, 20, 60), (60, 30, 30), (30, 30, 60), (80, 20, 20),
    (20, 60, 20), (20, 20, 80), (40, 40, 40), (100, 50, 50),
    (50, 50, 100), (30, 60, 60),
]

DARK_TEXT_PALETTE = [
    (0, 0, 0), (20, 20, 30), (40, 20, 20), (20, 40, 20),
    (20, 20, 60), (60, 30, 30), (30, 30, 60),
    (80, 20, 20), (20, 60, 20), (20, 20, 80),
]

LIGHT_TEXT_PALETTE = [
    (255, 255, 255), (240, 240, 240), (255, 230, 180),
    (255, 200, 200), (200, 230, 255), (230, 255, 200),
]


def sample_bg_color(rng, dark: bool):
    pal = DARK_BG_PALETTE if dark else LIGHT_BG_PALETTE
    base = rng.choice(pal)
    noise = rng.integers(-12, 13, size=3)
    return tuple(int(np.clip(c + n, 0, 255)) for c, n in zip(base, noise))


def sample_text_color(rng, dark_bg: bool):
    if dark_bg:
        pal = LIGHT_TEXT_PALETTE
    else:
        pal = DARK_TEXT_PALETTE
    base = rng.choice(pal)
    noise = rng.integers(-8, 9, size=3)
    return tuple(int(np.clip(c + n, 0, 255)) for c, n in zip(base, noise))


def color_contrast(c1, c2):
    return float(np.sqrt(sum((a - b) ** 2 for a, b in zip(c1, c2))))


# ============================================================
# ШРИФТЫ
# ============================================================
FONTS_DIR = Path("fonts")


def get_font_chars(font_path):
    try:
        tt = TTFont(str(font_path), fontNumber=0)
        return set(chr(c) for c in tt.getBestCmap().keys())
    except Exception:
        return set()


def font_supports_text(font_path, text):
    chars = get_font_chars(font_path)
    if not chars:
        return False
    return all(c in chars for c in text if c != " ")


def load_fonts():
    candidates = list(FONTS_DIR.rglob("*.ttf")) + list(FONTS_DIR.rglob("*.otf"))
    if not candidates:
        candidates = list(Path("fonts").rglob("*.ttf"))
    # Не фильтруем по REQUIRED_CHARS — проверка идёт на конкретный текст
    print(f"Найдено шрифтов: {len(candidates)}")
    return candidates


# ============================================================
# РЕНДЕР — теперь имитация бокса детектора
# ============================================================
def render_text_on_canvas(text, font_path, font_size, bg_color, text_color,
                          rng):
    """
    Рендерит текст на большом фоне, возвращает RGBA-картинку с текстом
    и координатами. Обрезка до бокса — отдельно.
    """
    try:
        font = ImageFont.truetype(str(font_path), size=font_size)
    except Exception:
        return None

    # Bbox текста
    try:
        bbox = font.getbbox(text)
    except Exception:
        return None
    tw = bbox[2] - bbox[0]
    th = bbox[3] - bbox[1]
    if tw <= 0 or th <= 0:
        return None

    # Холст существенно больше текста — поля будут отрезаться
    pad_x = int(th * rng.uniform(0.3, 1.2))
    pad_y = int(th * rng.uniform(0.2, 0.8))

    canvas_w = tw + 2 * pad_x
    canvas_h = th + 2 * pad_y

    img = Image.new("RGB", (canvas_w, canvas_h), bg_color)
    draw = ImageDraw.Draw(img)
    draw.text((pad_x - bbox[0], pad_y - bbox[1]), text,
              font=font, fill=text_color)

    # Координаты текста внутри холста
    text_box = (pad_x, pad_y, pad_x + tw, pad_y + th)
    return img, text_box


def crop_to_detector_box(img, text_box, rng):
    """
    Обрезает холст так, как это сделал бы детектор: с рандомными
    асимметричными полями, иногда чуть задевая текст.
    """
    W, H = img.size
    tx0, ty0, tx1, ty1 = text_box
    tw = tx1 - tx0
    th = ty1 - ty0

    # Поля: по бокам — маленькие, сверху/снизу — больше
    pad_left   = int(th * rng.uniform(0.0, 0.6))
    pad_right  = int(th * rng.uniform(0.0, 0.6))
    pad_top    = int(th * rng.uniform(0.0, 0.9))
    pad_bottom = int(th * rng.uniform(0.0, 0.9))

    # Иногда детектор задевает текст — отрицательный паддинг
    if rng.random() < 0.25:
        side = rng.choice(["l", "r", "t", "b"])
        if side == "l":
            pad_left = -int(tw * rng.uniform(0.02, 0.08))
        elif side == "r":
            pad_right = -int(tw * rng.uniform(0.02, 0.08))
        elif side == "t":
            pad_top = -int(th * rng.uniform(0.02, 0.08))
        else:
            pad_bottom = -int(th * rng.uniform(0.02, 0.08))

    x0 = max(0, tx0 - pad_left)
    y0 = max(0, ty0 - pad_top)
    x1 = min(W, tx1 + pad_right)
    y1 = min(H, ty1 + pad_bottom)

    if x1 - x0 < 12 or y1 - y0 < 8:
        return None
    return img.crop((x0, y0, x1, y1))


def resize_to_target_ar(img, target_ar):
    w, h = img.size
    new_w = max(10, int(h * target_ar))
    return img.resize((new_w, h), Image.BILINEAR)


def get_bg_color(img):
    """Цвет из угла — работает и на градиенте."""
    corners = [
        img.getpixel((0, 0)),
        img.getpixel((img.width - 1, 0)),
        img.getpixel((0, img.height - 1)),
        img.getpixel((img.width - 1, img.height - 1)),
    ]
    arr = np.array(corners, dtype=np.float32)
    return tuple(int(v) for v in np.median(arr, axis=0))


# ============================================================
# МЕТРИКИ
# ============================================================
def readability_score(img):
    arr = np.array(img.convert("L")).astype(np.float32)
    if arr.std() < 1e-3:
        return 0.0
    row_var = arr.var(axis=1).mean()
    col_var = arr.var(axis=0).mean()
    med = np.median(arr)
    active = (np.abs(arr - med) > 20).mean()
    return float(arr.std() * min(row_var, col_var) ** 0.5 * (active + 0.01))


def is_striped(img, ratio_threshold=0.03):
    arr = np.array(img.convert("L")).astype(np.float32)
    row_var = arr.var(axis=1).mean()
    col_var = arr.var(axis=0).mean()
    if row_var < 1e-3 or col_var < 1e-3:
        return True
    return min(row_var, col_var) / max(row_var, col_var) < ratio_threshold


def is_readable(img):
    """Смягчённые пороги — не отбрасываем низкоконтрастные кропы."""
    arr = np.array(img.convert("L")).astype(np.float32)
    if arr.std() < 8.0:
        return False
    med = np.median(arr)
    if (np.abs(arr - med) > 15).mean() < 0.005:
        return False
    return not is_striped(img)


def try_aug(img, aug_fn, rng, prob=1.0, min_read=0.8):
    if rng.random() > prob:
        return img
    before = readability_score(img)
    after = aug_fn(img, rng)
    if is_striped(after):
        return img
    after_score = readability_score(after)
    if before > 0 and after_score / before < min_read:
        return img
    return after


# ============================================================
# АУГМЕНТАЦИИ
# ============================================================
def aug_jpeg(img, rng):
    q = int(rng.integers(30, 86))
    buf = io.BytesIO()
    img.save(buf, "JPEG", quality=q)
    buf.seek(0)
    return Image.open(buf).convert("RGB")


def aug_contrast(img, rng):
    return ImageEnhance.Contrast(img).enhance(float(rng.uniform(0.85, 1.15)))


def aug_brightness(img, rng):
    return ImageEnhance.Brightness(img).enhance(float(rng.uniform(0.88, 1.12)))


def aug_color(img, rng):
    return ImageEnhance.Color(img).enhance(float(rng.uniform(0.6, 1.4)))


def aug_rotation_small(img, rng):
    """Мягкий наклон ±3° — как у реальных фото."""
    angle = float(rng.uniform(-3, 3))
    bg = get_bg_color(img)
    return img.rotate(angle, resample=Image.BILINEAR, fillcolor=bg, expand=False)


def aug_rotation_hard(img, rng):
    """Наклон ±5-8° — в пределах реальных тестовых кропов."""
    angle = float(rng.uniform(5, 8)) * rng.choice([-1, 1])
    bg = get_bg_color(img)
    return img.rotate(angle, resample=Image.BILINEAR, fillcolor=bg, expand=True)


def aug_crop(img, rng):
    """Независимая обрезка 0-10% с каждой стороны."""
    w, h = img.size
    cut_l = int(w * rng.uniform(0.0, 0.10))
    cut_r = int(w * rng.uniform(0.0, 0.10))
    cut_t = int(h * rng.uniform(0.0, 0.10))
    cut_b = int(h * rng.uniform(0.0, 0.10))
    x0, y0 = cut_l, cut_t
    x1, y1 = w - cut_r, h - cut_b
    if x1 - x0 < 12 or y1 - y0 < 8:
        return img
    return img.crop((x0, y0, x1, y1))


def aug_sensor_noise(img, rng):
    sigma = float(rng.uniform(4, 14))
    arr = np.array(img).astype(np.float32)
    arr += rng.normal(0, sigma, arr.shape)
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))


def aug_lighting_gradient(img, rng):
    w, h = img.size
    arr = np.array(img).astype(np.float32)
    coef_min = float(rng.uniform(0.75, 0.92))
    if rng.random() < 0.5:
        grad = np.linspace(coef_min, 1.0, w)
        arr *= grad[None, :, None]
    else:
        grad = np.linspace(coef_min, 1.0, h)
        arr *= grad[:, None, None]
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))


def aug_soft_blur(img, rng):
    r = float(rng.uniform(0.3, 1.2))
    return img.filter(ImageFilter.GaussianBlur(radius=r))


# ============================================================
# ПАЙПЛАЙН — независимая генерация каждого примера
# ============================================================
def generate_one(rng, fonts):
    text = sample_text(rng)

    # Шрифт подбираем под конкретный текст
    font_path = None
    for _ in range(20):
        candidate = rng.choice(fonts)
        if font_supports_text(candidate, text):
            font_path = candidate
            break
    if font_path is None:
        return None

    # Цвета: случайно светлый или тёмный фон
    dark_bg = rng.random() < P_DARK_BG
    bg_color = sample_bg_color(rng, dark_bg)
    text_color = sample_text_color(rng, dark_bg)

    if color_contrast(bg_color, text_color) < 120:
        text_color = (255, 255, 255) if dark_bg else (0, 0, 0)

    # Рендер на большом холсте
    font_size = int(rng.integers(14, 61))
    rendered = render_text_on_canvas(
        text, font_path, font_size, bg_color, text_color, rng
    )
    if rendered is None:
        return None
    img, text_box = rendered

    # Обрезка до бокса детектора
    img = crop_to_detector_box(img, text_box, rng)
    if img is None:
        return None

    # AR
    ar_choices = [2.5, 3.5, 4.5, 5.5, 7.0, 9.0, 12.0]
    ar_weights = [0.15, 0.20, 0.20, 0.15, 0.15, 0.10, 0.05]
    img = resize_to_target_ar(img, rng.choice(ar_choices, p=ar_weights))

    # JPEG-артефакты
    n_jpeg = int(rng.integers(1, 3))
    for _ in range(n_jpeg):
        img = aug_jpeg(img, rng)

    # Фотометрика
    if ENABLE_PHOTOMETRIC:
        if rng.random() < 0.30:
            img = try_aug(img, aug_contrast, rng, min_read=0.9)
        if rng.random() < 0.30:
            img = try_aug(img, aug_brightness, rng, min_read=0.9)
        if rng.random() < 0.25:
            img = try_aug(img, aug_color, rng, min_read=0.9)

    # Мягкий наклон ±3°
    if ENABLE_ROTATION and rng.random() < 0.40:
        img = try_aug(img, aug_rotation_small, rng, min_read=0.9)

    # Наклон ±5-8°
    if ENABLE_HARD_ROTATION and rng.random() < P_HARD_ROTATION:
        img = try_aug(img, aug_rotation_hard, rng, min_read=0.85)

    # Обрезка 0-10% независимо с каждой стороны
    if ENABLE_CROP and rng.random() < 0.40:
        img = try_aug(img, aug_crop, rng, min_read=0.7)

    # Шум
    if ENABLE_NOISE and rng.random() < 0.50:
        img = try_aug(img, aug_sensor_noise, rng, min_read=0.85)

    # Градиент
    if ENABLE_GRADIENT and rng.random() < 0.30:
        img = try_aug(img, aug_lighting_gradient, rng, min_read=0.9)

    # Blur — последним
    if ENABLE_SOFT_BLUR and rng.random() < 0.20:
        img = try_aug(img, aug_soft_blur, rng, min_read=0.9)

    if not is_readable(img):
        return None

    # Метка: 0 = 0°, 1 = 180°. Поворот ПОСЛЕ всех аугментаций.
    if rng.random() < 0.5:
        return img, 0
    return img.rotate(180, expand=False), 1


# ============================================================
# ГЕНЕРАЦИЯ ДАТАСЕТА
# ============================================================
def generate_dataset(n_samples, out_dir, fonts):
    out_dir = Path(out_dir)
    (out_dir / "0_degree").mkdir(parents=True, exist_ok=True)
    (out_dir / "180_degree").mkdir(parents=True, exist_ok=True)

    rng = np.random.default_rng(SEED)
    saved = 0
    attempts = 0
    max_attempts = n_samples * 5
    n_zero = 0
    n_one = 0

    pbar = tqdm(total=n_samples, desc="Generating")
    while saved < n_samples and attempts < max_attempts:
        attempts += 1
        res = generate_one(rng, fonts)
        if res is None:
            continue
        img, label = res
        sub = "0_degree" if label == 0 else "180_degree"
        img.save(out_dir / sub / f"{saved:08d}.png")
        saved += 1
        if label == 0:
            n_zero += 1
        else:
            n_one += 1
        pbar.update(1)
    pbar.close()
    print(f"Сохранено {saved} из {attempts} попыток "
          f"({100 * saved / max(attempts, 1):.1f}%)")
    print(f"Баланс: 0° = {n_zero}, 180° = {n_one}")


# ============================================================
# ЗАПУСК
# ============================================================
if __name__ == "__main__":
    print("=" * 60)
    print("Настройки v4:")
    print(f"  N_SAMPLES        = {N_SAMPLES}")
    print(f"  OUT_DIR          = {OUT_DIR}")
    print(f"  P_DARK_BG        = {P_DARK_BG}")
    print(f"  P_HARD_ROTATION  = {P_HARD_ROTATION}")
    print(f"  COLOR_BG         = {ENABLE_COLOR_BG}")
    print(f"  COLOR_TEXT       = {ENABLE_COLOR_TEXT}")
    print(f"  ROTATION small   = {ENABLE_ROTATION}")
    print(f"  ROTATION hard    = {ENABLE_HARD_ROTATION}")
    print(f"  CROP             = {ENABLE_CROP}")
    print(f"  INVERT           = {ENABLE_INVERSION}")
    print(f"  PHOTOMETRIC      = {ENABLE_PHOTOMETRIC}")
    print(f"  NOISE            = {ENABLE_NOISE}")
    print(f"  GRADIENT         = {ENABLE_GRADIENT}")
    print(f"  SOFT_BLUR        = {ENABLE_SOFT_BLUR}")
    print("=" * 60)

    fonts = load_fonts()
    assert len(fonts) >= 5, "Нужно минимум 5 шрифтов в fonts/"

    generate_dataset(N_SAMPLES, OUT_DIR, fonts)
    print(f"\nГотово: {OUT_DIR}/")

Настройки v4:
  N_SAMPLES        = 70000
  OUT_DIR          = data/v4
  P_DARK_BG        = 0.3
  P_HARD_ROTATION  = 0.25
  COLOR_BG         = True
  COLOR_TEXT       = True
  ROTATION small   = True
  ROTATION hard    = True
  CROP             = True
  INVERT           = True
  PHOTOMETRIC      = True
  NOISE            = True
  GRADIENT         = True
  SOFT_BLUR        = True
Найдено шрифтов: 15


Generating: 100%|██████████| 70000/70000 [09:55<00:00, 117.49it/s]

Сохранено 70000 из 70003 попыток (100.0%)
Баланс: 0° = 35034, 180° = 34966

Готово: data/v4/
